# Gateway Kartplex

## 100hz sensor data into one db

Laod the sqlite data export from the Sensor Logger app into duckdb. Add a race number column, join on time, round the seconds elapsed to 0.xx to match level of precision of the data (100hz), making it easier to read.

In [1]:
import os
import sqlite3
import pandas as pd
import duckdb
import numpy as np

# Path to the SQLite database
db_path = './gateway-1.sqlite'
output_db_path = 'sensors.duckdb'

# Define race variable
race = 1

def join_sensor_tables(db_path, output_db_path, race_id):
    # Check if file exists
    if not os.path.exists(db_path):
        print(f"Error: Database file '{db_path}' not found.")
        return

    # Connect to the SQLite database
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Tables to join
        tables_to_join = ['Orientation', 'Compass', 'Accelerometer', 'Gyroscope', 'Magnetometer']
        
        # Verify all tables exist and have time column (for joining)
        valid_tables = []
        join_column = None  # Will be set to 'time' or 'utc_time' depending on availability
        
        # First check if 'time' column exists in all tables
        all_have_time = True
        for table_name in tables_to_join:
            # Check if table exists
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name=?;", (table_name,))
            if cursor.fetchone() is None:
                print(f"Table '{table_name}' does not exist in the database.")
                all_have_time = False
                continue
                
            # Check if time column exists
            cursor.execute(f"PRAGMA table_info(\"{table_name}\")")
            columns = [col[1] for col in cursor.fetchall()]
            
            if "time" not in columns:
                print(f"Table '{table_name}' does not have a 'time' column.")
                all_have_time = False
                break
        
        # Decide which column to use for joining
        if all_have_time:
            join_column = "time"
            print("All tables have 'time' column, using it for joining.")
        else:
            join_column = "utc_time"
            print("Not all tables have 'time' column, falling back to 'utc_time' for joining.")
            
        # Now verify the chosen join column exists in all tables
        for table_name in tables_to_join:
            # Check if table exists
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name=?;", (table_name,))
            if cursor.fetchone() is None:
                continue
                
            # Check if join column exists
            cursor.execute(f"PRAGMA table_info(\"{table_name}\")")
            columns = [col[1] for col in cursor.fetchall()]
            
            if join_column not in columns:
                print(f"Table '{table_name}' does not have a '{join_column}' column.")
                continue
                
            valid_tables.append(table_name)
        
        if not valid_tables:
            print(f"No valid tables found to join on '{join_column}'.")
            return
            
        print(f"Found {len(valid_tables)} valid tables to join: {', '.join(valid_tables)}")
        
        # If using time for joining, check if time values match across tables for the same records
        if join_column == "time":
            # Check a small sample of data by joining on utc_time and comparing time values
            print("\nVerifying time values consistency across tables...")
            
            # Check if all tables have utc_time (to verify time consistency)
            all_have_utc_time = all("utc_time" in cursor.execute(f"PRAGMA table_info(\"{t}\")").fetchall() 
                                 for t in valid_tables)
                                 
            if all_have_utc_time:
                # Sample time values from first table
                sample_df = pd.read_sql_query(
                    f"SELECT time, utc_time FROM \"{valid_tables[0]}\" LIMIT 100", 
                    conn
                )
                
                # Check time values in other tables for the same utc_time
                for table_name in valid_tables[1:]:
                    other_sample = pd.read_sql_query(
                        f"SELECT time, utc_time FROM \"{table_name}\" LIMIT 100", 
                        conn
                    )
                    
                    # Merge on utc_time and check if time values match
                    merged = pd.merge(
                        sample_df, 
                        other_sample,
                        on="utc_time", 
                        suffixes=('_first', '_current')
                    )
                    
                    if len(merged) > 0 and not np.array_equal(merged['time_first'], merged['time_current']):
                        print(f"WARNING: time values differ between {valid_tables[0]} and {table_name} "
                              f"for the same utc_time values. Joining on time may produce incorrect results.")
            else:
                print("Not all tables have utc_time column, skipping time consistency check.")
        
        # Check if all tables have seconds_elapsed column and compare values
        seconds_elapsed_dfs = {}
        for table_name in valid_tables:
            cursor.execute(f"PRAGMA table_info(\"{table_name}\")")
            columns = [col[1] for col in cursor.fetchall()]
            
            if "seconds_elapsed" in columns:
                # Get sample of seconds_elapsed values with corresponding join column
                sample_df = pd.read_sql_query(
                    f"SELECT {join_column}, seconds_elapsed FROM \"{table_name}\" LIMIT 1000", 
                    conn
                )
                seconds_elapsed_dfs[table_name] = sample_df
        
        # Compare seconds_elapsed values across tables
        identical_seconds_elapsed = False
        if len(seconds_elapsed_dfs) > 1:
            print("\nChecking if seconds_elapsed columns are identical across tables...")
            
            # Use the first table as a reference
            reference_table = list(seconds_elapsed_dfs.keys())[0]
            identical_seconds_elapsed = True
            
            for table_name, df in seconds_elapsed_dfs.items():
                if table_name == reference_table:
                    continue
                
                # Merge reference table with current table on join column
                merged = pd.merge(
                    seconds_elapsed_dfs[reference_table], 
                    df,
                    on=join_column, 
                    suffixes=('_ref', '_current')
                )
                
                # Check if values are identical
                if not np.array_equal(merged['seconds_elapsed_ref'].round(3), 
                                     merged['seconds_elapsed_current'].round(3)):
                    identical_seconds_elapsed = False
                    print(f"seconds_elapsed values differ between {reference_table} and {table_name}")
                    break
            
            if identical_seconds_elapsed:
                print(f"All seconds_elapsed columns have identical values for the same {join_column}")
            else:
                print("seconds_elapsed columns differ between tables")
        
        # Build the SQL JOIN query
        table_columns = {}
        select_parts = []
        
        for table_name in valid_tables:
            cursor.execute(f"PRAGMA table_info(\"{table_name}\")")
            columns = [col[1] for col in cursor.fetchall()]
            table_columns[table_name] = columns
            
            # For the first table, include all columns
            if table_name == valid_tables[0]:
                # Round seconds_elapsed in the first table
                first_table_cols = []
                for col in columns:
                    if col == "seconds_elapsed":
                        first_table_cols.append(f"ROUND(t1.{col}, 2) AS {col}")
                    else:
                        first_table_cols.append(f"t1.{col}")
                select_parts.append(", ".join(first_table_cols))
            else:
                # For other tables, exclude join column to avoid duplicates
                # Also exclude seconds_elapsed if identical across all tables
                exclude_cols = [join_column]
                if identical_seconds_elapsed and "seconds_elapsed" in columns:
                    exclude_cols.append("seconds_elapsed")
                    
                # Also exclude utc_time if it exists and we're joining on time
                if join_column == "time" and "utc_time" in columns:
                    exclude_cols.append("utc_time")
                    
                other_cols = [col for col in columns if col not in exclude_cols]
                table_idx = valid_tables.index(table_name) + 1  # t1, t2, etc.
                
                if other_cols:
                    other_cols_parts = []
                    for col in other_cols:
                        if col == "seconds_elapsed":
                            other_cols_parts.append(f"ROUND(t{table_idx}.{col}, 3) AS {table_name}_{col}")
                        else:
                            other_cols_parts.append(f"t{table_idx}.{col} AS {table_name}_{col}")
                    
                    select_parts.append(", ".join(other_cols_parts))
        
        # Build the complete JOIN query
        select_clause = ", ".join(select_parts)
        from_clause = f" FROM \"{valid_tables[0]}\" t1"
        
        for i, table_name in enumerate(valid_tables[1:], 2):
            from_clause += f" INNER JOIN \"{table_name}\" t{i} ON t1.{join_column} = t{i}.{join_column}"
        
        query = f"SELECT {select_clause} {from_clause}"
        
        # Execute the query to get the joined data
        print(f"\nExecuting JOIN query on {join_column}...")
        print(f"Query: {query[:100]}...") # Print part of the query for debugging
        
        # Get the complete joined data as a DataFrame
        full_df = pd.read_sql_query(query, conn)
        
        # Add race column to the DataFrame
        full_df['race'] = race_id
        print(f"Added 'race' column with value {race_id}")
        
        # Close the SQLite connection
        conn.close()
        
        # Double-check that seconds_elapsed is properly rounded
        if 'seconds_elapsed' in full_df.columns:
            # Ensure seconds_elapsed is properly rounded to 3 decimal places
            full_df['seconds_elapsed'] = full_df['seconds_elapsed'].round(3)
            print("Rounded seconds_elapsed to 3 decimal places")
        
        # Check for any other seconds_elapsed columns from other tables
        for col in full_df.columns:
            if col.endswith('_seconds_elapsed'):
                full_df[col] = full_df[col].round(3)
                print(f"Rounded {col} to 3 decimal places")
        
        # Reorder columns to put race, seconds_elapsed, utc_time, time first
        cols = full_df.columns.tolist()
        priority_cols = []
        
        # First add race
        if 'race' in cols:
            priority_cols.append('race')
            cols.remove('race')
        
        # Then add seconds_elapsed
        if 'seconds_elapsed' in cols:
            priority_cols.append('seconds_elapsed')
            cols.remove('seconds_elapsed')
        
        # Then add utc_time
        if 'utc_time' in cols:
            priority_cols.append('utc_time')
            cols.remove('utc_time')
        
        # Then add time
        if 'time' in cols:
            priority_cols.append('time')
            cols.remove('time')
        
        # Reorder the DataFrame
        full_df = full_df[priority_cols + cols]
        print(f"Columns reordered: {', '.join(priority_cols)} now appear first")
        
        # Connect to the output DuckDB database (create if not exists)
        print(f"Saving joined data to DuckDB database: {output_db_path}")
        duck_conn = duckdb.connect(output_db_path)
        
        # Drop the existing table if it exists
        duck_conn.execute("DROP TABLE IF EXISTS combined_sensors")
        
        # Register the DataFrame with DuckDB
        duck_conn.register('full_df_view', full_df)
        
        # Create the table from the registered view
        duck_conn.execute("CREATE TABLE combined_sensors AS SELECT * FROM full_df_view")
        
        # Verify the column order and rounding
        columns = duck_conn.execute("PRAGMA table_info(combined_sensors)").fetchall()
        column_names = [col[1] for col in columns]
        
        print(f"\nColumns in combined_sensors table (first few): {column_names[:10]}")
        
        # Check seconds_elapsed formatting
        if 'seconds_elapsed' in column_names:
            seconds_sample = duck_conn.execute("SELECT seconds_elapsed FROM combined_sensors LIMIT 5").fetchall()
            print(f"Sample seconds_elapsed values: {seconds_sample}")
        
        # Get row count to confirm
        result = duck_conn.execute("SELECT COUNT(*) FROM combined_sensors").fetchone()
        row_count = result[0]
        
        # Verify race column values
        if 'race' in column_names:
            race_values = duck_conn.execute("SELECT DISTINCT race FROM combined_sensors").fetchall()
            print(f"Distinct values in race column: {race_values}")
        
        print(f"Data successfully saved to {output_db_path} in table 'combined_sensors'")
        print(f"Total rows saved: {row_count}")
        
        # Close the DuckDB connection
        duck_conn.close()
        
    except sqlite3.Error as e:
        print(f"SQLite error: {e}")
    except Exception as e:
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()
    finally:
        if 'conn' in locals():
            conn.close()


join_sensor_tables(db_path, output_db_path, race)

All tables have 'time' column, using it for joining.
Found 5 valid tables to join: Orientation, Compass, Accelerometer, Gyroscope, Magnetometer

Verifying time values consistency across tables...
Not all tables have utc_time column, skipping time consistency check.

Checking if seconds_elapsed columns are identical across tables...
All seconds_elapsed columns have identical values for the same time

Executing JOIN query on time...
Query: SELECT t1.time, ROUND(t1.seconds_elapsed, 2) AS seconds_elapsed, t1.yaw, t1.qx, t1.qz, t1.roll, t1.q...
Added 'race' column with value 1
Rounded seconds_elapsed to 3 decimal places
Columns reordered: race, seconds_elapsed, utc_time, time now appear first
Saving joined data to DuckDB database: sensors.duckdb

Columns in combined_sensors table (first few): ['race', 'seconds_elapsed', 'utc_time', 'time', 'yaw', 'qx', 'qz', 'roll', 'qw', 'qy']
Sample seconds_elapsed values: [(0.1,), (0.11,), (0.12,), (0.13,), (0.14,)]
Distinct values in race column: [(1,)]

In [2]:
import duckdb
import pandas as pd

def check_sensors_db(db_path='sensors.duckdb'):
    try:
        # Connect to the DuckDB database
        print(f"Connecting to DuckDB database: {db_path}")
        conn = duckdb.connect(db_path)
        
        # Get column names
        columns_query = "SELECT column_name FROM information_schema.columns WHERE table_name='combined_sensors'"
        columns = [row[0] for row in conn.execute(columns_query).fetchall()]
        
        print("\nTable columns:")
        for i, col in enumerate(columns, 1):
            print(f"{i}. {col}")
        
        # Count rows
        count_query = "SELECT COUNT(*) FROM combined_sensors"
        row_count = conn.execute(count_query).fetchone()[0]
        print(f"\nTotal rows in database: {row_count}")
        
        if row_count > 0:
            # Get first row
            first_row_query = "SELECT * FROM combined_sensors ORDER BY utc_time ASC LIMIT 1"
            first_row = conn.execute(first_row_query).fetchone()
            
            # Get last row
            last_row_query = "SELECT * FROM combined_sensors ORDER BY utc_time DESC LIMIT 1"
            last_row = conn.execute(last_row_query).fetchone()
            
            # Find the index of utc_time column
            utc_time_idx = columns.index('utc_time')
            
            # Print utc_time for first and last row
            print(f"\nFirst row utc_time: {first_row[utc_time_idx]}")
            print(f"Last row utc_time: {last_row[utc_time_idx]}")
            
            # Calculate time span
            if isinstance(first_row[utc_time_idx], (int, float)) and isinstance(last_row[utc_time_idx], (int, float)):
                time_span = last_row[utc_time_idx] - first_row[utc_time_idx]
                print(f"Time span: {time_span} seconds ({time_span/60:.2f} minutes)")
            
        else:
            print("Database is empty.")
            
    except Exception as e:
        print(f"Error: {str(e)}")
    finally:
        if 'conn' in locals():
            conn.close()

if __name__ == "__main__":
    check_sensors_db()

Connecting to DuckDB database: sensors.duckdb

Table columns:
1. race
2. seconds_elapsed
3. utc_time
4. time
5. yaw
6. qx
7. qz
8. roll
9. qw
10. qy
11. pitch
12. Compass_magneticBearing
13. Accelerometer_z
14. Accelerometer_y
15. Accelerometer_x
16. Gyroscope_z
17. Gyroscope_y
18. Gyroscope_x
19. Magnetometer_z
20. Magnetometer_y
21. Magnetometer_x

Total rows in database: 135529

First row utc_time: 2025-10-02T21:46:18.666Z
Last row utc_time: 2025-10-02T22:09:01.394Z


In [3]:
import duckdb
import pandas as pd

def load_sensors_db(db_path='sensors.duckdb'):
    try:
        # Connect to the DuckDB database
        print(f"Connecting to DuckDB database: {db_path}")
        conn = duckdb.connect(db_path)
        
        # Query all data from the combined_sensors table
        print("Loading data into DataFrame...")
        query = "SELECT * FROM combined_sensors"
        df = conn.execute(query).fetchdf()
        
        # Show basic information about the DataFrame
        print(f"\nDataFrame loaded successfully with {len(df)} rows and {len(df.columns)} columns")
        print(f"Memory usage: {df.memory_usage().sum() / (1024**2):.2f} MB")
        
        # Display DataFrame info
        print("\nDataFrame information:")
        print(df.info())
        
        # Show first few rows
        print("\nFirst 5 rows:")
        print(df.head())
        
        # Close the connection
        conn.close()
        
        return df
        
    except Exception as e:
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

if __name__ == "__main__":
    df = load_sensors_db()
    print("\nDataFrame shape:", df.shape)
    
    # Additional operations you might want to do with the DataFrame
    # print("\nColumn names:", df.columns.tolist())
    # print("\nData types:", df.dtypes)
    # print("\nSummary statistics:")
    print(df.describe())

Connecting to DuckDB database: sensors.duckdb
Loading data into DataFrame...

DataFrame loaded successfully with 135529 rows and 21 columns
Memory usage: 21.71 MB

DataFrame information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135529 entries, 0 to 135528
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   race                     135529 non-null  int64  
 1   seconds_elapsed          135529 non-null  float64
 2   utc_time                 135529 non-null  object 
 3   time                     135529 non-null  int64  
 4   yaw                      135529 non-null  float64
 5   qx                       135529 non-null  float64
 6   qz                       135529 non-null  float64
 7   roll                     135529 non-null  float64
 8   qw                       135529 non-null  float64
 9   qy                       135529 non-null  float64
 10  pitch                    135529 non-n

## GPS data

Commercial grade 1hz GPS data from iPhone. Interpolation with IMU data will be used to create the traces after loading and cleaning the GPS data.

Altitude measurements from the Location table where GPS data is stored can be set to 0 and it will be relative to the flat surface

This code defines a function migrate_location_table that transfers GPS location data from a SQLite database to a DuckDB database while cleaning and transforming the data. Here's a breakdown of what it does:

Data Extraction:

Connects to a SQLite database and reads the "Location" table into a pandas DataFrame
Data Cleaning:

Removes rows with negative seconds_elapsed values
Sorts data chronologically by time
Detects and removes GPS anomalies by calculating speeds between consecutive points and removing unrealistic values (over 50 m/s or ~180 km/h)
Resets the time sequence (seconds_elapsed) to start from 0 and increment sequentially
Data Transformation:

Sets all altitude values to 0 meters for flat surface visualization
Rounds numeric columns for better precision:
Coordinates to 5 decimal places
Speed to 3 decimal places
Accuracy and bearing measurements to 2 decimal places
Reorders columns to prioritize time, seconds_elapsed, speed, latitude, and longitude
Data Export:

Saves the cleaned data to a CSV file
Creates a new table in DuckDB from the CSV data
Verifies the data was loaded correctly by counting rows
When the script is run directly, it processes a specific SQLite database file ('gateway-1.sqlite'), converts it as described above, and stores the results in both a CSV file ('gps.csv') and a DuckDB database ('gps.duckdb').

This appears to be a data migration tool specifically for GPS track data, possibly for race kart analysis, focusing on cleaning unrealistic data points and standardizing the format.

In [4]:
from haversine import haversine

def migrate_location_table(sqlite_path, duckdb_path, csv_path):
    # Connect to SQLite database
    print(f"Connecting to SQLite database at {sqlite_path}...")
    sqlite_conn = sqlite3.connect(sqlite_path)
    
    # Read data using pandas
    print("Reading Location table from SQLite...")
    df = pd.read_sql_query("SELECT * FROM Location", sqlite_conn)
    
    if 'seconds_elapsed' in df.columns:
        # Remove rows with negative seconds_elapsed
        print(f"Removing rows with negative seconds_elapsed... (before: {len(df)} rows)")
        df = df[df['seconds_elapsed'] >= 0]
        print(f"After removal: {len(df)} rows")
        
        # Sort by time to ensure chronological order
        print("Sorting data chronologically...")
        df = df.sort_values('seconds_elapsed')
        
        # Detect and remove GPS anomalies
        print("Detecting and removing GPS anomalies...")
        # Calculate distance between consecutive points
        from haversine import haversine
        
        distances = []
        anomaly_indices = []
        
        for i in range(1, len(df)):
            prev_point = (df.iloc[i-1]['latitude'], df.iloc[i-1]['longitude'])
            curr_point = (df.iloc[i]['latitude'], df.iloc[i]['longitude'])
            distance = haversine(prev_point, curr_point, unit='m')  # distance in meters
            distances.append(distance)
            
            # Calculate time difference in seconds
            time_diff = df.iloc[i]['seconds_elapsed'] - df.iloc[i-1]['seconds_elapsed']
            
            # If time_diff is too small, avoid division by zero
            if time_diff < 0.01:
                time_diff = 0.01
                
            # Calculate speed in m/s
            speed = distance / time_diff
            
            # Detect anomalies: points with unrealistic speeds (e.g., > 100 m/s or ~360 km/h)
            # Adjust this threshold based on your specific scenario
            if speed > 50:  # 50 m/s is about 180 km/h, adjust as needed for kart racing
                anomaly_indices.append(i)
        
        if anomaly_indices:
            print(f"Found {len(anomaly_indices)} anomalous points. Removing them...")
            df = df.drop(df.index[anomaly_indices])
            print(f"After anomaly removal: {len(df)} rows")
        
        # Reset the seconds_elapsed values: first row to 0, then incrementing by 1
        print("Resetting seconds_elapsed values...")
        df = df.sort_values('seconds_elapsed')  # Ensure data is sorted
        df['seconds_elapsed'] = range(len(df))  # Assign 0, 1, 2, 3, ...
        df['seconds_elapsed'] = df['seconds_elapsed'].astype(float)  # Convert to float
        
        # Rest of your processing remains the same
        print("Setting all altitude values to 0 meters for flat surface visualization...")
        if 'altitude' in df.columns:
            df['altitude'] = 0.0
        if 'altitudeAboveMeanSeaLevel' in df.columns:
            df['altitudeAboveMeanSeaLevel'] = 0.0
            
        # Rounding numeric columns
        # ... rest of your rounding code ...
        
        # Coordinates (5 decimal places ≈ 1.1m precision)
        if 'latitude' in df.columns:
            df['latitude'] = df['latitude'].round(5)
        if 'longitude' in df.columns:
            df['longitude'] = df['longitude'].round(5)
            
        # Speed (3 decimal places for m/s)
        if 'speed' in df.columns:
            df['speed'] = df['speed'].round(3)
            
        # Accuracy measurements (2 decimal places)
        if 'horizontalAccuracy' in df.columns:
            df['horizontalAccuracy'] = df['horizontalAccuracy'].round(2)
        if 'verticalAccuracy' in df.columns:
            df['verticalAccuracy'] = df['verticalAccuracy'].round(2)
        if 'speedAccuracy' in df.columns:
            df['speedAccuracy'] = df['speedAccuracy'].round(2)
        if 'bearingAccuracy' in df.columns:
            df['bearingAccuracy'] = df['bearingAccuracy'].round(2)
            
        # Bearing/heading (2 decimal places)
        if 'bearing' in df.columns:
            df['bearing'] = df['bearing'].round(2)
        
        # Reorder columns as requested
        print("Reordering columns...")
        cols = ['time', 'seconds_elapsed', 'speed', 'latitude', 'longitude']
        remaining_cols = [col for col in df.columns if col not in cols]
        ordered_cols = cols + remaining_cols
        ordered_cols = [col for col in ordered_cols if col in df.columns]
        df = df[ordered_cols]
    else:
        print("Warning: seconds_elapsed column not found")
    
    
    # Save to CSV
    print(f"Saving data to {csv_path}...")
    df.to_csv(csv_path, index=False)
    print(f"Data saved to CSV. Row count: {len(df)}")
    
    # Connect to DuckDB
    print(f"Connecting to DuckDB at {duckdb_path}...")
    duck_conn = duckdb.connect(duckdb_path)
    
    # Create table from CSV
    print("Creating Location table in DuckDB from CSV...")
    duck_conn.execute("DROP TABLE IF EXISTS Location")
    duck_conn.execute(f"CREATE TABLE Location AS SELECT * FROM read_csv_auto('{csv_path}')")
    
    # Verify the data was loaded
    result = duck_conn.execute("SELECT COUNT(*) FROM Location").fetchone()
    print(f"Data loaded into DuckDB. Row count: {result[0]}")
    
    # Close connections
    sqlite_conn.close()
    duck_conn.close()
    
    print("Migration completed successfully!")
    
if __name__ == "__main__":
    sqlite_db_path = './gateway-1.sqlite'
    duckdb_path = './gps.duckdb'
    csv_path = './gps.csv'
    migrate_location_table(sqlite_db_path, duckdb_path, csv_path)

Connecting to SQLite database at ./gateway-1.sqlite...
Reading Location table from SQLite...
Removing rows with negative seconds_elapsed... (before: 1364 rows)
After removal: 1363 rows
Sorting data chronologically...
Detecting and removing GPS anomalies...
Resetting seconds_elapsed values...
Setting all altitude values to 0 meters for flat surface visualization...
Reordering columns...
Saving data to ./gps.csv...
Data saved to CSV. Row count: 1363
Connecting to DuckDB at ./gps.duckdb...
Creating Location table in DuckDB from CSV...
Data loaded into DuckDB. Row count: 1363
Migration completed successfully!


# Location precision

GPS location data is much lower frequency than the sensor data, noisy and not very precise since it's consumer gps. This can be addressed with IMU/GPS fusion techniques. 

Enhanced Precision Approach with IMU Data:
Extended Kalman Filter (EKF) or Complementary Filter

Integrate GPS (low frequency, absolute position) with IMU (high frequency, relative movement)
Can achieve 10-30cm precision in racing contexts
Handles the strengths/weaknesses of each sensor type
Dead Reckoning Between GPS Fixes

Use double integration of accelerometer data for position changes
Use gyroscope for orientation tracking
Reset accumulated error at each GPS fix
Particularly valuable for capturing racing line nuances between GPS samples
Motion Model Constraints

Apply vehicle dynamics constraints (maximum lateral g-forces, etc.)
Helps correct integration drift from accelerometer data
Advanced Implementations

SLAM (Simultaneous Localization and Mapping) techniques
Particle filters for multimodal uncertainty representation
Error-state Kalman filters specifically designed for IMU-GPS fusion
Implementation Strategy:
Pre-process IMU data:

Apply bias correction and calibration
Filter for noise reduction (low-pass filter)
Convert to world coordinate frame
GPS-IMU Time Synchronization:

Align timestamps between systems
Account for any latency in GPS data
Fusion Algorithm:

Process IMU data at full 100Hz
Incorporate GPS fixes when available
Use GPS accuracy values as covariance weights
Post-processing:

Apply physics-based smoothing
Racing line optimization
With 100Hz IMU data and GPS together, you can achieve centimeter-level precision for racing applications, capturing every nuance of braking points, turn-in, apex, and track-out positions. This is comparable to professional motorsport telemetry systems.

This code defines a function migrate_and_fuse_imu_data that extracts, processes, and combines Inertial Measurement Unit (IMU) data from a SQLite database into a DuckDB database. Here's what it does:

Data Extraction:

Connects to a SQLite database
Reads two separate tables: "Accelerometer" and "Gyroscope" into pandas DataFrames
These contain motion sensor data from an IMU device
Time Processing:

Converts time columns to integers (nanoseconds)
Calculates seconds_elapsed based on the minimum time across both datasets
Converts nanosecond timestamps to human-readable UTC time strings
Data Export for Debugging:

Saves the individual Accelerometer and Gyroscope tables to CSV files
DuckDB Database Creation:

Creates individual tables for Accelerometer and Gyroscope data in DuckDB
Data Fusion:

Creates a new combined table called "IMU_Fused" by joining the accelerometer and gyroscope data on the exact same timestamps
The fused table contains:
Time information (raw time, seconds elapsed, UTC time)
Accelerometer data (x, y, z axes)
Gyroscope data (x, y, z axes)
Orders the data chronologically
Verification and Export:

Checks and reports the row counts of all tables
Exports the fused IMU data to a CSV file called "imu.csv"
When run directly, the script processes data from a file named 'gateway.sqlite' and stores the results in 'imu.duckdb' and CSV files.

This script is designed for sensor data processing, likely for motion analysis or tracking applications. It takes raw sensor data from two separate sources (accelerometer and gyroscope) and aligns them by timestamp to create a unified dataset that contains complete motion information at each point in time.

In [5]:
import sqlite3
import duckdb
import pandas as pd
import os
import numpy as np
from datetime import datetime, timezone

def migrate_and_fuse_imu_data(sqlite_path, duckdb_path):
    # Connect to SQLite database
    print(f"Connecting to SQLite database at {sqlite_path}...")
    sqlite_conn = sqlite3.connect(sqlite_path)
    
    # Read Accelerometer data using pandas
    print("Reading Accelerometer table from SQLite...")
    accel_df = pd.read_sql_query("SELECT * FROM Accelerometer", sqlite_conn)
    print(f"Accelerometer data loaded. Row count: {len(accel_df)}")
    
    # Read Gyroscope data using pandas
    print("Reading Gyroscope table from SQLite...")
    gyro_df = pd.read_sql_query("SELECT * FROM Gyroscope", sqlite_conn)
    print(f"Gyroscope data loaded. Row count: {len(gyro_df)}")
    
    # Close SQLite connection
    sqlite_conn.close()
    
    # Convert time columns to integers for joining (assuming they're in nanoseconds)
    print("Converting time columns to integers for joining...")
    accel_df['time_int'] = accel_df['time'].astype('int64')
    gyro_df['time_int'] = gyro_df['time'].astype('int64')
    
    # Calculate seconds_elapsed and format utc_time
    print("Processing time-related columns...")
    min_time = min(accel_df['time_int'].min(), gyro_df['time_int'].min())
    
    # For accelerometer data
    accel_df['seconds_elapsed'] = ((accel_df['time_int'] - min_time) / 1e9).round(2)
    
    # Convert nanosecond timestamps to UTC time strings
    def ns_to_utc_str(ns_timestamp):
        seconds = ns_timestamp / 1e9
        dt = datetime.fromtimestamp(seconds, tz=timezone.utc)
        return dt.strftime('%Y-%m-%dT%H:%M:%S.%fZ')
    
    accel_df['utc_time'] = accel_df['time_int'].apply(ns_to_utc_str)
    gyro_df['seconds_elapsed'] = ((gyro_df['time_int'] - min_time) / 1e9).round(2)
    gyro_df['utc_time'] = gyro_df['time_int'].apply(ns_to_utc_str)
    
    # Save the individual tables to CSV for potential debugging
    accel_df.to_csv('./Accelerometer.csv', index=False)
    gyro_df.to_csv('./Gyroscope.csv', index=False)
    
    # Connect to DuckDB
    print(f"Connecting to DuckDB at {duckdb_path}...")
    duck_conn = duckdb.connect(duckdb_path)
    
    # Create individual tables in DuckDB
    print("Creating tables in DuckDB...")
    duck_conn.execute("DROP TABLE IF EXISTS Accelerometer")
    duck_conn.execute("CREATE TABLE Accelerometer AS SELECT * FROM read_csv_auto('./Accelerometer.csv')")
    
    duck_conn.execute("DROP TABLE IF EXISTS Gyroscope")
    duck_conn.execute("CREATE TABLE Gyroscope AS SELECT * FROM read_csv_auto('./Gyroscope.csv')")
    
    # Create fused IMU table by joining on time_int
    print("Creating fused IMU table by joining Accelerometer and Gyroscope on time_int...")
    duck_conn.execute("""
    DROP TABLE IF EXISTS IMU_Fused;
    CREATE TABLE IMU_Fused AS
    SELECT 
        a.time_int,
        a.time,
        a.seconds_elapsed,
        a.utc_time,
        a.x as accel_x,
        a.y as accel_y,
        a.z as accel_z,
        g.x as gyro_x,
        g.y as gyro_y,
        g.z as gyro_z
    FROM 
        Accelerometer a
    INNER JOIN 
        Gyroscope g
    ON 
        a.time_int = g.time_int
    ORDER BY 
        a.time_int
    """)
    
   # Verify the data was loaded and joined correctly
    accel_count = duck_conn.execute("SELECT COUNT(*) FROM Accelerometer").fetchone()[0]
    gyro_count = duck_conn.execute("SELECT COUNT(*) FROM Gyroscope").fetchone()[0]
    fused_count = duck_conn.execute("SELECT COUNT(*) FROM IMU_Fused").fetchone()[0]
    
    print(f"Data loaded into DuckDB:")
    print(f"  - Accelerometer rows: {accel_count}")
    print(f"  - Gyroscope rows: {gyro_count}")
    print(f"  - Fused IMU rows: {fused_count}")
    
    # Save the fused IMU data to CSV
    print("Saving fused IMU data to imu.csv...")
    duck_conn.execute("COPY (SELECT * FROM IMU_Fused) TO './imu.csv' (HEADER, DELIMITER ',')")
    print("Fused IMU data saved to imu.csv")
    
    # Close connection
    duck_conn.close()
    
    print("Migration and fusion completed successfully!")

if __name__ == "__main__":
    sqlite_db_path = './gateway.sqlite'
    duckdb_path = './imu.duckdb'
    
    migrate_and_fuse_imu_data(sqlite_db_path, duckdb_path)

Connecting to SQLite database at ./gateway.sqlite...
Reading Accelerometer table from SQLite...


DatabaseError: Execution failed on sql 'SELECT * FROM Accelerometer': no such table: Accelerometer

## Create race trace using fancy algo with GPS and IMU

This code implements a sensor fusion algorithm that combines GPS and Inertial Measurement Unit (IMU) data to create a high-precision race track trajectory. Here's what it does:

Data Processing and Fusion:

Takes GPS data (position, speed) and IMU data (acceleration, gyroscope readings)
Uses advanced filtering and signal processing techniques to combine these data sources
Creates a smoother, more accurate trajectory than either sensor could provide alone
Key Functions:

butter_lowpass_filter: Removes high-frequency noise from sensor data
lat_lon_to_meters and meters_to_lat_lon: Convert between GPS coordinates and local XY coordinates
integrate_imu_motion: Uses IMU data to estimate motion between GPS samples
fuse_gps_imu: The main function that combines everything
Fusion Process:

Converts GPS coordinates to local meters for easier calculations
Creates a smooth base trajectory from GPS using spline interpolation
Uses IMU data to fill in details between GPS points (which typically come at 1Hz)
Blends GPS and IMU data with adaptive weights (trusts GPS more near GPS samples)
Calculates additional metrics like lateral/longitudinal acceleration and turn rate
Technical Approaches:

Uses dead reckoning between GPS fixes
Applies complementary filtering to combine GPS and IMU data
Uses splines for smoothness and Butterworth filtering to reduce noise
Output:

Creates a high-frequency trajectory dataset (at IMU sample rate, ~100Hz)
Includes position, speed, bearing, acceleration, and turn rate
Formats timestamps for compatibility with Kepler visualization
Generates race statistics (average speed, max speed, max acceleration, etc.)
This is essentially a specialized algorithm for race track telemetry analysis, providing high-precision trajectory data that can be used to analyze driving performance, racing lines, and vehicle dynamics. The algorithm overcomes limitations of GPS (low sample rate, inaccuracy in turns) and IMU (drift over time) by combining their complementary strengths.

In [ ]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d, UnivariateSpline
from scipy.signal import butter, filtfilt

def butter_lowpass_filter(data, cutoff, fs, order=4):
    """Apply low-pass Butterworth filter to reduce noise."""
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

def lat_lon_to_meters(lat, lon, ref_lat, ref_lon):
    """Convert lat/lon to local x/y meters from reference point."""
    # Approximate meters per degree at reference latitude
    lat_m_per_deg = 111132.92 - 559.82 * np.cos(2 * np.radians(ref_lat)) + 1.175 * np.cos(4 * np.radians(ref_lat))
    lon_m_per_deg = 111412.84 * np.cos(np.radians(ref_lat)) - 93.5 * np.cos(3 * np.radians(ref_lat))
    
    x = (lon - ref_lon) * lon_m_per_deg
    y = (lat - ref_lat) * lat_m_per_deg
    return x, y

def meters_to_lat_lon(x, y, ref_lat, ref_lon):
    """Convert local x/y meters back to lat/lon."""
    lat_m_per_deg = 111132.92 - 559.82 * np.cos(2 * np.radians(ref_lat)) + 1.175 * np.cos(4 * np.radians(ref_lat))
    lon_m_per_deg = 111412.84 * np.cos(np.radians(ref_lat)) - 93.5 * np.cos(3 * np.radians(ref_lat))
    
    lat = ref_lat + (y / lat_m_per_deg)
    lon = ref_lon + (x / lon_m_per_deg)
    return lat, lon

def integrate_imu_motion(imu_df, dt=0.01):
    """
    Use IMU data to estimate relative motion between GPS samples.
    Returns velocity and displacement estimates.
    """
    # Rotate accelerometer data to account for device orientation
    # Assuming phone is mounted flat, accel_x = forward/back, accel_y = left/right
    accel_forward = imu_df['accel_x'].values
    accel_lateral = imu_df['accel_y'].values
    gyro_yaw = imu_df['gyro_z'].values  # Yaw rate (turning)
    
    # Remove gravity bias and filter accelerometer data
    # Filter high-frequency noise
    accel_forward_filtered = butter_lowpass_filter(accel_forward, cutoff=10, fs=100)
    accel_lateral_filtered = butter_lowpass_filter(accel_lateral, cutoff=10, fs=100)
    gyro_yaw_filtered = butter_lowpass_filter(gyro_yaw, cutoff=10, fs=100)
    
    # Integrate gyro to get heading changes
    heading_change = np.cumsum(gyro_yaw_filtered * dt)
    
    # Integrate acceleration to get velocity (will drift, but useful between GPS fixes)
    vel_forward = np.cumsum(accel_forward_filtered * dt)
    vel_lateral = np.cumsum(accel_lateral_filtered * dt)
    
    return {
        'heading_change': heading_change,
        'vel_forward': vel_forward,
        'vel_lateral': vel_lateral,
        'accel_forward': accel_forward_filtered,
        'accel_lateral': accel_lateral_filtered,
        'gyro_yaw': gyro_yaw_filtered
    }

def fuse_gps_imu(gps_file, imu_file, output_file='racetrace.csv'):
    """
    Fuse GPS and IMU data for improved position accuracy and smooth curves.
    """
    print("Reading GPS data...")
    gps_df = pd.read_csv(gps_file)
    
    print("Reading IMU data...")
    imu_df = pd.read_csv(imu_file)
    
    # Time vectors
    gps_time = gps_df['seconds_elapsed'].values
    imu_time = imu_df['seconds_elapsed'].values
    dt = np.mean(np.diff(imu_time))  # IMU sample period
    
    print(f"GPS: {len(gps_df)} points at 1Hz")
    print(f"IMU: {len(imu_df)} points at {1/dt:.1f}Hz")
    print(f"Duration: {imu_time[-1] - imu_time[0]:.2f} seconds")
    
    # Convert GPS to local coordinates (meters)
    ref_lat = gps_df['latitude'].mean()
    ref_lon = gps_df['longitude'].mean()
    gps_x, gps_y = lat_lon_to_meters(
        gps_df['latitude'].values,
        gps_df['longitude'].values,
        ref_lat, ref_lon
    )
    
    print(f"\nReference point: {ref_lat:.6f}, {ref_lon:.6f}")
    print(f"Track dimensions: {gps_x.max()-gps_x.min():.1f}m x {gps_y.max()-gps_y.min():.1f}m")
    
    # Get GPS speed and compute heading from position changes
    gps_speed = gps_df['speed'].values
    gps_bearing = gps_df['bearing'].values
    
    # Handle -1.0 bearing values by computing from position
    dx = np.diff(gps_x)
    dy = np.diff(gps_y)
    computed_bearing = np.degrees(np.arctan2(dx, dy)) % 360
    computed_bearing = np.concatenate([[computed_bearing[0]], computed_bearing])
    
    # Use computed bearing where GPS bearing is invalid
    gps_bearing = np.where(gps_bearing < 0, computed_bearing, gps_bearing)
    
    print("\nProcessing IMU data...")
    imu_motion = integrate_imu_motion(imu_df, dt)
    
    # Step 1: Interpolate GPS positions with spline for smoothness
    print("Creating smooth GPS trajectory...")
    
    # Use spline interpolation for smoother curves
    # Lower smoothing factor = closer to GPS points, higher = smoother
    spline_x = UnivariateSpline(gps_time, gps_x, s=10, k=3)
    spline_y = UnivariateSpline(gps_time, gps_y, s=10, k=3)
    
    # Get base smooth trajectory
    smooth_x = spline_x(imu_time)
    smooth_y = spline_y(imu_time)
    
    # Step 2: Use IMU to refine position between GPS fixes
    print("Fusing IMU data for improved precision...")
    
    fused_x = np.zeros_like(smooth_x)
    fused_y = np.zeros_like(smooth_y)
    
    # Interpolate GPS bearing and speed to IMU rate
    interp_bearing = interp1d(gps_time, gps_bearing, kind='linear', 
                              bounds_error=False, fill_value='extrapolate')(imu_time)
    interp_speed = interp1d(gps_time, gps_speed, kind='linear',
                           bounds_error=False, fill_value='extrapolate')(imu_time)
    
    # Estimate heading from spline + IMU gyro
    base_heading = interp_bearing
    heading = np.zeros_like(base_heading)
    heading[0] = base_heading[0]
    
    # Blend GPS heading with IMU gyro integration
    alpha = 0.98  # Weight for GPS heading (higher = trust GPS more)
    
    for i in range(1, len(imu_time)):
        # Find nearest GPS sample
        gps_idx = np.argmin(np.abs(gps_time - imu_time[i]))
        time_since_gps = abs(imu_time[i] - gps_time[gps_idx])
        
        # Closer to GPS fix = trust GPS more, further away = use IMU more
        if time_since_gps < 0.5:
            blend = alpha + (1 - alpha) * (time_since_gps * 2)
        else:
            blend = 0.5
        
        # Integrate gyro for heading change
        imu_heading_change = imu_motion['gyro_yaw'][i] * dt * 57.2958  # rad/s to deg
        heading[i] = blend * base_heading[i] + (1 - blend) * (heading[i-1] + imu_heading_change)
    
    # Compute velocity components from speed and heading
    heading_rad = np.radians(heading)
    vx = interp_speed * np.sin(heading_rad)
    vy = interp_speed * np.cos(heading_rad)
    
    # Initialize first position from smooth trajectory
    fused_x[0] = smooth_x[0]
    fused_y[0] = smooth_y[0]
    
    # Forward pass: integrate velocity with GPS corrections
    for i in range(1, len(imu_time)):
        # Dead reckoning from IMU
        dr_x = fused_x[i-1] + vx[i] * dt
        dr_y = fused_y[i-1] + vy[i] * dt
        
        # Blend with smooth GPS trajectory
        # More trust in GPS at GPS sample times
        gps_idx = np.argmin(np.abs(gps_time - imu_time[i]))
        time_since_gps = abs(imu_time[i] - gps_time[gps_idx])
        
        if time_since_gps < 0.05:  # Very close to GPS sample
            weight_gps = 0.95
        elif time_since_gps < 0.3:  # Within 300ms
            weight_gps = 0.7
        else:  # Between GPS samples
            weight_gps = 0.4
        
        fused_x[i] = weight_gps * smooth_x[i] + (1 - weight_gps) * dr_x
        fused_y[i] = weight_gps * smooth_y[i] + (1 - weight_gps) * dr_y
    
    # Apply smoothing to remove jitter
    fused_x = butter_lowpass_filter(fused_x, cutoff=5, fs=100)
    fused_y = butter_lowpass_filter(fused_y, cutoff=5, fs=100)
    
    # Convert back to lat/lon
    print("Converting back to latitude/longitude...")
    fused_lat, fused_lon = meters_to_lat_lon(fused_x, fused_y, ref_lat, ref_lon)
    
    # Compute fused speed and bearing from positions
    dx_fused = np.diff(fused_x)
    dy_fused = np.diff(fused_y)
    fused_speed = np.sqrt(dx_fused**2 + dy_fused**2) / dt
    fused_speed = np.concatenate([[fused_speed[0]], fused_speed])
    fused_speed = butter_lowpass_filter(fused_speed, cutoff=3, fs=100)
    
    fused_bearing = np.degrees(np.arctan2(dx_fused, dy_fused)) % 360
    fused_bearing = np.concatenate([[fused_bearing[0]], fused_bearing])
    
    # Compute lateral and longitudinal acceleration
    accel_long = np.diff(fused_speed) / dt
    accel_long = np.concatenate([[0], accel_long])
    
    # Lateral acceleration from speed and turn rate
    turn_rate = imu_motion['gyro_yaw'] * 57.2958  # rad/s to deg/s
    accel_lat = fused_speed * np.radians(turn_rate)
    
    # Create output dataframe
    print("Creating output dataframe...")
    output_df = imu_df.copy()
    
    # Ensure proper timestamps for Kepler
    # Kepler recognizes Unix timestamps in MILLISECONDS for timeline (not seconds)
    if 'time' in output_df.columns:
        # Convert nanosecond timestamps to milliseconds
        timestamp_ms = (output_df['time'] / 1e6).astype(np.int64)
    elif 'time_int' in output_df.columns:
        timestamp_ms = (output_df['time_int'] / 1e6).astype(np.int64)
    else:
        # If no time column, create from elapsed time
        timestamp_ms = ((imu_time + gps_df['time'].iloc[0] / 1e6)).astype(np.int64)
    
    # Store as integer milliseconds - Kepler needs this for timeline
    output_df['timestamp'] = timestamp_ms
    
    # Also keep seconds_elapsed for reference
    output_df['seconds_elapsed'] = imu_time
    
    # Create properly formatted UTC datetime strings
    # Kepler expects ISO 8601 format: YYYY-MM-DDTHH:MM:SS.sssZ
    from datetime import datetime
    utc_times = [datetime.utcfromtimestamp(ts / 1000.0).strftime('%Y-%m-%dT%H:%M:%S.%f')[:-3] + 'Z' 
                 for ts in timestamp_ms]
    output_df['utc_time'] = utc_times
    
    # Add fused GPS data
    output_df['latitude'] = fused_lat
    output_df['longitude'] = fused_lon
    output_df['speed'] = fused_speed
    output_df['bearing'] = fused_bearing
    output_df['heading'] = heading
    
    # Add computed metrics
    output_df['accel_longitudinal'] = accel_long
    output_df['accel_lateral'] = accel_lat
    output_df['turn_rate'] = turn_rate
    
    # Add position in meters for analysis
    output_df['x_meters'] = fused_x
    output_df['y_meters'] = fused_y
    
    # Interpolate other GPS fields
    for col in ['altitude', 'horizontalAccuracy', 'verticalAccuracy']:
        if col in gps_df.columns:
            interp_func = interp1d(gps_time, gps_df[col].values, kind='linear',
                                  bounds_error=False, fill_value='extrapolate')
            output_df[col] = interp_func(imu_time)
    
    # Save output
    print(f"\nSaving to {output_file}...")
    
    # Reorder columns to put timestamp first for Kepler
    cols = output_df.columns.tolist()
    if 'timestamp' in cols:
        cols.remove('timestamp')
        cols = ['timestamp'] + cols
        output_df = output_df[cols]
    
    output_df.to_csv(output_file, index=False)
    
    print(f"✓ Complete! Created {output_file} with {len(output_df)} rows")
    print(f"\nTimestamp range: {output_df['timestamp'].iloc[0]} to {output_df['timestamp'].iloc[-1]}")
    print(f"Time delta between samples: {np.mean(np.diff(output_df['timestamp'].values)):.2f} ms")
    
    # Statistics
    print("\n=== Race Statistics ===")
    print(f"Duration: {imu_time[-1] - imu_time[0]:.2f} seconds")
    print(f"Average speed: {fused_speed.mean():.2f} m/s ({fused_speed.mean()*2.237:.2f} mph)")
    print(f"Max speed: {fused_speed.max():.2f} m/s ({fused_speed.max()*2.237:.2f} mph)")
    print(f"Max lateral accel: {np.abs(accel_lat).max():.2f} m/s² ({np.abs(accel_lat).max()/9.81:.2f}g)")
    print(f"Max longitudinal accel: {accel_long.max():.2f} m/s² ({accel_long.max()/9.81:.2f}g)")
    print(f"Max braking: {accel_long.min():.2f} m/s² ({abs(accel_long.min())/9.81:.2f}g)")
    print(f"Max turn rate: {np.abs(turn_rate).max():.1f} deg/s")
    
    return output_df

if __name__ == "__main__":
    # Example usage
    gps_file = "gps.csv"
    imu_file = "imu.csv"
    
    df = fuse_gps_imu(gps_file, imu_file, output_file='racetrace.csv')
    
    print("\n=== Sample Output ===")
    print(df[['seconds_elapsed', 'latitude', 'longitude', 'speed', 'bearing', 
              'accel_longitudinal', 'accel_lateral', 'turn_rate']].head(10))

## Start/Finish Laps
This code processes race track data to identify and label individual laps in a racing session. Here's what it does:

Define Start/Finish Line:

Sets up fixed GPS coordinates for the left and right edges of the start/finish line
Creates a LineString object representing this line
Load and Process Track Data:

Reads the previously generated 'racetrace.csv' file (from the sensor fusion code)
Iterates through each GPS position in the track data
Detect Lap Crossings:

For each consecutive pair of GPS points, creates a line segment
Checks if this segment intersects with the start/finish line
Increments the lap counter each time a crossing is detected
Label and Categorize Laps:

Adds a 'lap' column to the dataframe containing the lap number for each data point
Creates a 'lap_name' column that categorizes laps as:
'out': Initial exit from pits (lap 0)
'warm-up': First lap (lap 1)
'timed': Regular racing laps (middle laps)
'exit': Final lap (last lap number)
Renumber Laps:

Recategorizes the lap numbering scheme
Sets all non-racing laps (out/warm-up/exit) to lap 0
Renumbers the timed laps starting from 1
Output:

Prints statistics about the detected laps
Saves the enhanced dataset to 'racetrace_with_laps.csv'
This script is useful for race analysis as it separates the race into meaningful segments - the out lap, warm-up lap, timed/racing laps, and the in lap. This enables better analysis of racing performance by focusing on the actual racing laps and excluding the non-competitive portions.

The defined start/finish line coordinates (38.64886, -90.13461) to (38.64888, -90.13440) appear to be at a specific race track (likely in the St. Louis area based on the coordinates).


In [ ]:
import pandas as pd
import numpy as np
from shapely.geometry import LineString, Point

# Define the start/finish line coordinates
sf_left = (38.64886, -90.13461)
sf_right = (38.64888, -90.13440)

# Create a LineString representing the start/finish line
sf_line = LineString([sf_left, sf_right])

# Load the CSV file
df = pd.read_csv('racetrace.csv')

# Initialize lap counter and create new columns for lap numbers
lap_number = 0
lap_numbers = []

# Previous point to check if we've crossed the line
prev_point = None

for i, row in df.iterrows():
    current_point = Point(row['latitude'], row['longitude'])
    
    # For the first point, just record it and continue
    if prev_point is None:
        prev_point = current_point
        lap_numbers.append(lap_number)
        continue
    
    # Create a line segment from previous point to current point
    segment = LineString([prev_point, current_point])
    
    # Check if this segment intersects with the start/finish line
    if segment.intersects(sf_line):
        lap_number += 1
    
    lap_numbers.append(lap_number)
    prev_point = current_point

# Add the lap numbers to the dataframe
df['lap'] = lap_numbers

# Get total number of laps for later reference
total_laps = lap_number

# Add lap name column
def get_lap_name(lap_num):
    if lap_num == 0:
        return 'out'
    elif lap_num == 1:
        return 'warm-up'
    elif lap_num == total_laps:
        return 'exit'
    else:
        return 'timed'

df['lap_name'] = df['lap'].apply(get_lap_name)

# Renumber laps: out/warm-up/exit are 0, timed laps are numbered 1, 2, 3, etc.
def renumber_lap(row):
    if row['lap_name'] in ['out', 'warm-up', 'exit']:
        return 0
    else:
        # For timed laps, subtract 1 from original lap number to start at 1
        # (since warm-up was lap 1, first timed lap was 2, which becomes 1)
        return row['lap'] - 1

df['numbered_lap'] = df.apply(renumber_lap, axis=1)
df['lap'] = df['numbered_lap']
df = df.drop(columns=['numbered_lap'])

# Display some information about the laps
print(f"Total laps detected: {total_laps}")
print(df.groupby(['lap', 'lap_name']).size())

# Save the result
df.to_csv('racetrace_with_laps.csv', index=False)

# Best possible times

Total lap time with best sectors

The data was off for the sectors, so had to recalculate. Need to fix that in the dataset.

This code performs a racing telemetry analysis focused on sector and lap times. Here's what it does:

Data Loading and Preparation:

Loads a CSV file containing racing telemetry data from Circuit of the Americas (COTA)
Makes a copy of the original data and ensures sectors are in numeric format
Sorts the data chronologically by lap, sector, and elapsed time
Sector Time Analysis:

Groups data by lap and sector to find the start and end time of each sector
Calculates the actual duration of each sector (end_time - start_time)
Creates a summary table of sector boundaries and durations
Theoretical Best Lap Calculation:

Identifies the fastest time achieved for each sector across all laps
Sums these fastest sector times to create a "theoretical best lap" time
This represents the potential lap time if a driver drove each sector at their best
Fastest Sector Details:

For each sector, identifies which lap contained the fastest sector time
Displays the fastest sector times along with the corresponding lap numbers
Actual Lap Time Calculation:

Computes the total time for each lap by summing its sector times
Identifies the fastest complete lap (a lap with all sectors completed)
Formats the time in minutes:seconds.milliseconds for readability
Performance Analysis:

Calculates the potential improvement by comparing the best actual lap with the theoretical best lap
The difference indicates how much time could potentially be gained with more consistent driving
Summary Output:

Displays a formatted summary of all lap times for comparison
Shows the theoretical best lap time and the actual best lap time
This analysis is valuable for race drivers or teams to identify areas for improvement. By breaking down performance by sectors, it helps pinpoint specific parts of the track where the driver is losing time compared to their best performance in those sectors.

In [ ]:
import pandas as pd
import numpy as np

# Load the telemetry data
df = pd.read_csv('./cota-4-telemetry.csv')

# Make a copy of the original dataframe
df_original = df.copy()

# Check relevant columns
print("Data preview:")
print(df[['time', 'race_seconds_elapsed', 'lap', 'sector', 'sector_time', 'lap_seconds_elapsed']].head())

# Convert sector to numeric if needed
if df['sector'].dtype == 'object':
    df['sector'] = pd.to_numeric(df['sector'], errors='coerce')

# Sort data by lap, sector, and elapsed time to ensure chronological order
df.sort_values(['lap', 'sector', 'race_seconds_elapsed'], inplace=True)

# Get the first and last timestamp for each lap-sector combination
sector_boundaries = df.groupby(['lap', 'sector']).agg({
    'race_seconds_elapsed': ['min', 'max']
}).reset_index()

# Flatten the column hierarchy
sector_boundaries.columns = ['lap', 'sector', 'start_time', 'end_time']

# Calculate the actual sector duration
sector_boundaries['calculated_sector_time'] = sector_boundaries['end_time'] - sector_boundaries['start_time']

print("\nSector time calculation check:")
print(sector_boundaries[['lap', 'sector', 'calculated_sector_time']].head(10))

# Find the fastest time for each sector across all laps
fastest_sectors = sector_boundaries.groupby('sector')['calculated_sector_time'].min().reset_index()

# Calculate the theoretical best lap time
best_lap_time = fastest_sectors['calculated_sector_time'].sum()

# Convert to minutes:seconds.milliseconds format
minutes = int(best_lap_time // 60)
seconds = int(best_lap_time % 60)
milliseconds = int((best_lap_time % 1) * 1000)
best_lap_time_formatted = f"{minutes}:{seconds:02d}.{milliseconds:03d}"

# Show detailed results
print("\nFastest sector times (recalculated):")
for _, row in fastest_sectors.iterrows():
    # Find the lap where this fastest sector occurred
    fastest_lap = sector_boundaries[
        (sector_boundaries['sector'] == row['sector']) & 
        (sector_boundaries['calculated_sector_time'] == row['calculated_sector_time'])
    ]['lap'].values[0]
    print(f"Sector {row['sector']}: {row['calculated_sector_time']:.3f}s (from lap {fastest_lap})")

print(f"\nTheoretical best lap time: {best_lap_time:.3f}s ({best_lap_time_formatted})")

# Calculate actual lap times by summing sector times for each lap
lap_times = sector_boundaries.groupby('lap')['calculated_sector_time'].sum().reset_index()
lap_times = lap_times.rename(columns={'calculated_sector_time': 'total_lap_time'})

# Find the best actual lap time
if not lap_times.empty:
    # Filter out laps with missing sectors
    complete_laps = lap_times[
        lap_times['lap'].isin(
            sector_boundaries.groupby('lap').size()[
                sector_boundaries.groupby('lap').size() == len(fastest_sectors)
            ].index
        )
    ]
    
    if not complete_laps.empty:
        best_actual_lap = complete_laps['total_lap_time'].min()
        best_actual_lap_number = complete_laps.loc[complete_laps['total_lap_time'].idxmin(), 'lap']

        minutes = int(best_actual_lap // 60)
        seconds = int(best_actual_lap % 60)
        milliseconds = int((best_actual_lap % 1) * 1000)
        best_actual_lap_formatted = f"{minutes}:{seconds:02d}.{milliseconds:03d}"

        print(f"\nBest actual lap time: {best_actual_lap:.3f}s ({best_actual_lap_formatted}) on lap {best_actual_lap_number}")
        print(f"Potential improvement: {(best_actual_lap - best_lap_time):.3f}s")
    else:
        print("\nNo complete laps found with all sectors")
else:
    print("\nCouldn't calculate actual lap times - data may be incomplete")

# Print a summary of all lap times for comparison
print("\nAll lap times (recalculated):")
for _, row in lap_times.iterrows():
    minutes = int(row['total_lap_time'] // 60)
    seconds = int(row['total_lap_time'] % 60)
    milliseconds = int((row['total_lap_time'] % 1) * 1000)
    lap_time_formatted = f"{minutes}:{seconds:02d}.{milliseconds:03d}"
    print(f"Lap {row['lap']}: {row['total_lap_time']:.3f}s ({lap_time_formatted})")

This code performs an advanced multi-race telemetry analysis to identify the ultimate theoretical best lap time across multiple racing sessions. Here's what it does:

Multi-Race Data Processing:

Defines four race files to analyze (gateway-1 through gateway-4)
Creates a function process_race_file to handle each race file consistently
Per-Race Analysis (for each race file):

Loads telemetry data and extracts the race number from the filename
Sorts data chronologically by lap, sector, and elapsed time
Calculates sector boundaries and durations
Finds the fastest time for each sector within that race
Calculates a theoretical best lap time for that race by combining its fastest sectors
Computes actual lap times and identifies the best complete lap
Cross-Race Analysis:

Combines sector data from all races into consolidated datasets
Identifies the ultimate fastest time for each sector across ALL races
For each fastest sector, tracks which race it came from
Calculates an "ultimate theoretical best lap time" by combining the fastest sectors from any race
Comparative Analysis:

Compares best actual lap times from each race
Identifies the best overall actual lap time and which race it came from
Calculates the potential improvement between the best actual lap and the ultimate theoretical best
Results Presentation:

Displays the ultimate fastest sectors with their source races
Shows the ultimate theoretical best lap time
Lists the best actual lap times by race
Identifies the best overall actual lap with its race and lap number
Quantifies the potential improvement between actual performance and theoretical best
Lists theoretical best times for each individual race
This code is particularly useful for analyzing a driver's performance across multiple races at the same track. It helps identify which race had the best performance in each sector, the overall best lap time, and how much potential improvement is possible if the driver could combine their best sector performances into a single lap.

In [ ]:
import pandas as pd
import numpy as np
import os
import glob

# List of race files to analyze
race_files = [
    './gateway-1-telemetry.csv',
    './gateway-2-telemetry.csv',
    './gateway-3-telemetry.csv',
    './gateway-4-telemetry.csv'
]

# Function to process a single race file
def process_race_file(file_path):
    print(f"\nProcessing {os.path.basename(file_path)}...")
    
    # Load the telemetry data
    df = pd.read_csv(file_path)
    
    # Extract race number from filename
    race_number = os.path.basename(file_path).split('-')[1]
    
    # Convert sector to numeric if needed
    if df['sector'].dtype == 'object':
        df['sector'] = pd.to_numeric(df['sector'], errors='coerce')
    
    # Sort data by lap, sector, and elapsed time to ensure chronological order
    df.sort_values(['lap', 'sector', 'race_seconds_elapsed'], inplace=True)
    
    # Get the first and last timestamp for each lap-sector combination
    sector_boundaries = df.groupby(['lap', 'sector']).agg({
        'race_seconds_elapsed': ['min', 'max']
    }).reset_index()
    
    # Flatten the column hierarchy
    sector_boundaries.columns = ['lap', 'sector', 'start_time', 'end_time']
    
    # Calculate the actual sector duration
    sector_boundaries['calculated_sector_time'] = sector_boundaries['end_time'] - sector_boundaries['start_time']
    
    # Add race identifier
    sector_boundaries['race'] = race_number
    
    # Find the fastest time for each sector in this race
    race_fastest_sectors = sector_boundaries.groupby('sector')['calculated_sector_time'].min().reset_index()
    race_fastest_sectors['race'] = race_number
    
    # Calculate best theoretical lap time for this race
    race_best_lap_time = race_fastest_sectors['calculated_sector_time'].sum()
    
    # Calculate actual lap times by summing sector times for each lap
    lap_times = sector_boundaries.groupby('lap')['calculated_sector_time'].sum().reset_index()
    lap_times = lap_times.rename(columns={'calculated_sector_time': 'total_lap_time'})
    lap_times['race'] = race_number
    
    # Find the best actual lap time in this race
    if not lap_times.empty:
        # Filter out laps with missing sectors
        sector_count = len(race_fastest_sectors)
        complete_laps = lap_times[
            lap_times['lap'].isin(
                sector_boundaries.groupby('lap').size()[
                    sector_boundaries.groupby('lap').size() == sector_count
                ].index
            )
        ]
        
        if not complete_laps.empty:
            best_actual_lap = complete_laps['total_lap_time'].min()
            best_actual_lap_number = complete_laps.loc[complete_laps['total_lap_time'].idxmin(), 'lap']
        else:
            best_actual_lap = None
            best_actual_lap_number = None
    else:
        best_actual_lap = None
        best_actual_lap_number = None
    
    race_summary = {
        'race': race_number,
        'sector_data': sector_boundaries,
        'fastest_sectors': race_fastest_sectors,
        'theoretical_best': race_best_lap_time,
        'best_actual_lap': best_actual_lap,
        'best_actual_lap_number': best_actual_lap_number,
        'lap_times': lap_times
    }
    
    return race_summary

# Process each race file
all_race_data = []
for file in race_files:
    try:
        race_data = process_race_file(file)
        all_race_data.append(race_data)
        print(f"Successfully processed {file}")
    except Exception as e:
        print(f"Error processing {file}: {e}")

# Combine sector data from all races
all_sectors = pd.concat([race['sector_data'] for race in all_race_data])
all_fastest_sectors = pd.concat([race['fastest_sectors'] for race in all_race_data])

# Find the fastest time for each sector across ALL races
ultimate_fastest_sectors = all_fastest_sectors.groupby('sector')['calculated_sector_time'].min().reset_index()

# For each sector in ultimate_fastest_sectors, find which race it came from
for idx, row in ultimate_fastest_sectors.iterrows():
    sector = row['sector']
    time = row['calculated_sector_time']
    source_race = all_fastest_sectors[(all_fastest_sectors['sector'] == sector) & 
                                     (all_fastest_sectors['calculated_sector_time'] == time)]['race'].values[0]
    ultimate_fastest_sectors.at[idx, 'source_race'] = source_race

# Calculate the ultimate theoretical best lap time
ultimate_best_lap_time = ultimate_fastest_sectors['calculated_sector_time'].sum()

# Convert to minutes:seconds.milliseconds format
minutes = int(ultimate_best_lap_time // 60)
seconds = int(ultimate_best_lap_time % 60)
milliseconds = int((ultimate_best_lap_time % 1) * 1000)
best_lap_time_formatted = f"{minutes}:{seconds:02d}.{milliseconds:03d}"

# Print results
print("\n" + "="*50)
print("ULTIMATE FASTEST SECTORS ACROSS ALL RACES")
print("="*50)
for _, row in ultimate_fastest_sectors.iterrows():
    print(f"Sector {row['sector']}: {row['calculated_sector_time']:.3f}s (from Race {row['source_race']})")

print(f"\nULTIMATE THEORETICAL BEST LAP TIME: {ultimate_best_lap_time:.3f}s ({best_lap_time_formatted})")
print("="*50)

# Compare with best actual laps from each race
print("\nBEST ACTUAL LAP TIMES BY RACE:")
print("-"*50)
best_actual_overall = float('inf')
best_race_number = None
best_lap_number = None

for race in all_race_data:
    if race['best_actual_lap'] is not None:
        race_num = race['race']
        lap_time = race['best_actual_lap']
        lap_num = race['best_actual_lap_number']
        
        minutes = int(lap_time // 60)
        seconds = int(lap_time % 60)
        milliseconds = int((lap_time % 1) * 1000)
        lap_time_formatted = f"{minutes}:{seconds:02d}.{milliseconds:03d}"
        
        print(f"Race {race_num}: {lap_time:.3f}s ({lap_time_formatted}) on lap {lap_num}")
        
        if lap_time < best_actual_overall:
            best_actual_overall = lap_time
            best_race_number = race_num
            best_lap_number = lap_num

if best_race_number is not None:
    minutes = int(best_actual_overall // 60)
    seconds = int(best_actual_overall % 60)
    milliseconds = int((best_actual_overall % 1) * 1000)
    best_overall_formatted = f"{minutes}:{seconds:02d}.{milliseconds:03d}"
    
    print("\nBEST OVERALL ACTUAL LAP:")
    print(f"{best_actual_overall:.3f}s ({best_overall_formatted}) on Race {best_race_number}, Lap {best_lap_number}")
    print(f"Potential improvement: {(best_actual_overall - ultimate_best_lap_time):.3f}s")

# Print individual race theoretical best times
print("\nTHEORETICAL BEST TIMES BY RACE:")
print("-"*50)
for race in all_race_data:
    race_num = race['race']
    theo_time = race['theoretical_best']
    
    minutes = int(theo_time // 60)
    seconds = int(theo_time % 60)
    milliseconds = int((theo_time % 1) * 1000)
    theo_time_formatted = f"{minutes}:{seconds:02d}.{milliseconds:03d}"
    
    print(f"Race {race_num}: {theo_time:.3f}s ({theo_time_formatted})")